**Notebook: The First LLM Call (Module 4)**
- OCBC AI Creator Masterclass, Day 1

___

This notebook is the hands-on part of Module 4 (about 31 minutes). After it come the example script
`day1/examples/m4_example.py` and the exercise `day1/exercises/m4_exercise.py`. Run it from the top.

Every LLM call uses quota. Run each LLM cell **once**; if you hit a quota error, change `OPENAI_MODEL` in `.env`.

# Background

The Module 3 report arrives at 07:30. The RMs say: *"We don't read tables at 07:45. Give us four sentences."*
Your manager wants the LLM to write that digest, but **only from our report**:

- **4.1 What an LLM call is.** Messages in, one message out, and nothing remembered.
- **4.2 Calling the model from code.** The reply, the tokens, and a conversation you keep yourself.
- **4.3 The report in the prompt.** Grounding the model in our data.
- **4.4 Prompt templates.** One prompt with blanks, filled for any report or bank.
- **4.5 Structured output.** Fields that code can use, checked by Pydantic.

| Step on the ladder | Who decides WHICH data | Who decides WHEN |
|---|---|---|
| **Report in the prompt (Modules 3–4)** | **Our code** | **The scheduler** |
| RAG (Module 5) | The user's question | The user |

## Training Objectives

- Send rules and a prompt to an OpenAI model through Pydantic AI and read the reply and its token use
- Explain why the model must get every fact in the prompt, and keep a conversation yourself
- Put a report in the prompt so the model answers from it, and write rules that stop invented numbers
- Turn a prompt into a reusable template with blanks and send it to the model
- Get structured output as a Pydantic object, and design your own schema

## The report

So that everyone works with the same numbers, this notebook builds the Module 3 report from the saved Sectors
answers in `data/sample/` (prices up to **23 October 2026**). The scheduled job uses today's report.

## Checking your answers

**Your turn** cells say what the right result is. Run the next cell once for the **Knowledge Check** questions.

In [ ]:
import hashlib

ANSWERS = {'m4_roles': 'df099a9916', 'm4_stateless': '732060d870', 'm4_tags': 'e5aaa8f5b0', 'm4_missing': '5cb1874458', 'm4_template': 'afe6d5a967', 'm4_braces': '89167596c9', 'm4_literal': 'e9b702fbe5', 'm4_threshold': 'fa4e8c23ee'}

def check_answer(question, answer):
    key = hashlib.sha256(f"{question}:{answer.strip().lower()}".encode()).hexdigest()[:10]
    if answer.strip() == "":
        print("Type a, b, c or d between the quotes, then run the cell again.")
    elif key == ANSWERS[question]:
        print("Correct.")
    else:
        print("Not quite. Read the question again and try another answer.")

In [ ]:
import os
from dotenv import load_dotenv

# Work from the project root (the folder with data/), so file paths are the same as in the .py scripts
while not os.path.exists("data/sample"):
    os.chdir("..")

load_dotenv()  # reads the keys from .env
print("Working folder:", os.getcwd())

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown

TICKERS = ["BBCA", "BBRI", "BMRI", "NISP"]

def load_sample(name):
    with open(f"data/sample/{name}.json", encoding="utf-8") as f:
        return json.load(f)

lines = ["# IDX Bank Report, 2026-10-23", "",
         "| Ticker | Company | Last close | Date | 1-day | 10-day | Volume vs avg |", "|---|---|---|---|---|---|---|"]
for t in TICKERS:
    prices = pd.DataFrame(load_sample(f"daily_{t}")).tail(10)
    close = prices["close"]
    lines.append(f"| {t} | {load_sample(f'report_{t}')['company_name']} | {close.iloc[-1]:,} | {prices['date'].iloc[-1]} | "
                 f"{close.iloc[-1] / close.iloc[-2] - 1:+.2%} | {close.iloc[-1] / close.iloc[0] - 1:+.2%} | "
                 f"{prices['volume'].iloc[-1] / prices['volume'].mean():.2f}x |")
report = "\n".join(lines)
Markdown(report)

# 4.1 What an LLM Call Is

> **The request**
>
> Your manager: **"Before we hand the model our report, show me what a call actually sends and gets back."**

> **In this section**
>
> - **Key question:** What goes into one call, and what comes back?
> - **How we run it:** **Live, about 5 minutes.**
> - **What you will do:** create the model and an agent, send the rules and a prompt, and read the reply.
> - **What carries forward:** `model`, used in every section below.

A call sends a list of **messages** and gets one message back. In Pydantic AI an **`Agent`** wraps the model:
its **`instructions`** hold the rules (the *system* message), the prompt we pass to `run` holds the task and the data
(the *user* message), and the reply comes back in **`result.output`**. `result.all_messages()` shows what was sent
and received. The key and the endpoint come from `.env`.

In a notebook we `await agent.run(...)`; in a .py script the same call is `agent.run_sync(...)`.

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider

model = OpenAIResponsesModel(
    os.getenv("OPENAI_MODEL", "gpt-6.1-sol"),
    provider=OpenAIProvider(api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("OPENAI_URL")),
)

agent = Agent(model, instructions="You answer questions about Indonesian bank stocks in one sentence. Never give buy or sell advice.")
result = await agent.run("What kind of company is PT Bank OCBC NISP Tbk?")
print(type(result).__name__)
print(result.output)

#### Knowledge Check: Where do the rules go?

The digest must never contain buy or sell advice, for every RM and every morning. Where does that rule belong?

- **a.** In the prompt, after the report
- **b.** In the agent's `instructions` (the system message)
- **c.** In the model name
- **d.** In `.env`

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m4_roles", answer)

# 4.2 Calling the Model from Code

> **The request**
>
> Your manager: **"What does a call cost, and can the RM ask a follow-up question?"**

> **In this section**
>
> - **Key question:** What do we read from the reply, and how does the model 'remember'?
> - **How we run it:** **Live, about 6 minutes.**
> - **What you will do:** read the text and the token use, see that a second call remembers nothing, and pass the history yourself.
> - **What carries forward:** the habit of sending everything the model needs in every call.

Read the reply with **`result.output`**. `result.usage` shows the **tokens**, which is what you pay for.
Then a second, separate call:

In [ ]:
print("Tokens:", result.usage)

follow_up = await agent.run("What is its ticker on the Indonesia Stock Exchange?")
print(follow_up.output)

The model has no idea what "its" means: every call is **stateless**. A chat app "remembers" because it sends
the earlier messages again, with the model's own answers. `result.all_messages()` is that list, and
`message_history=` sends it again.

#### Your turn: Keep the conversation yourself

Send the follow-up question again, but this time with the history: `message_history=result.all_messages()` (the
4.1 messages and the model's answer). Print the reply and the total tokens (`.usage.total_tokens`). The right
result names the ticker `NISP`, and uses more tokens than the first call, because the history is sent again.

In [ ]:
## Your code below



#### Knowledge Check: Stateless

A scheduled job asks the model for the digest at 07:30, and at 07:35 a second job asks "Which bank fell most in
the digest?" without sending anything else. What happens?

- **a.** The model answers from the 07:30 digest
- **b.** The model cannot know: it gets no digest in the second call
- **c.** Pydantic AI sends the digest automatically
- **d.** The model provider caches the digest for an hour

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m4_stateless", answer)

# 4.3 The Report in the Prompt

> **The request**
>
> Your manager: **"The digest must use our numbers, not whatever the model thinks it knows."**

> **In this section**
>
> - **Key question:** How do we make the model answer from our data, and only from it?
> - **How we run it:** **Live, about 7 minutes.**
> - **What you will do:** ask without data, then with the report between tags and strict rules; then write a prompt of your own.
> - **What carries forward:** `SYSTEM_PROMPT`, `digest_agent` and the `<report>` pattern, used in the example and the exercise.

In [ ]:
no_data = await Agent(model).run("How did BBCA close on 23 October 2026? Give the price.")
print(no_data.output)

Without data the model either refuses or, worse, gives a number that looks right. With the report **between
tags**, the data is clearly separated from our instructions, and the rules (the instructions of `digest_agent`) say to use nothing else:

In [ ]:
SYSTEM_PROMPT = """You write a short morning digest of Indonesian bank stocks for relationship managers.
Use only the facts in the report. Quote numbers exactly, with their dates.
Never give buy, sell or hold advice, price targets or predictions."""

human_prompt = f"""<report>
{report}
</report>

Write the digest: one headline sentence, then one short sentence per bank."""

digest_agent = Agent(model, instructions=SYSTEM_PROMPT)
digest = await digest_agent.run(human_prompt)
print(digest.output)

#### Your turn: A one-bank brief

An RM covering BBCA wants **two sentences about BBCA only**: the last close with its date, and the 1-day and 10-day
change. Write the prompt (the report between `<report>` tags, then the task), run it with `digest_agent` (the rules in
`SYSTEM_PROMPT`), and print the reply. The right result quotes `5,025` on `2026-10-23`, `-2.90%` and `-15.55%`, and gives no advice.

In [ ]:
## Your code below



#### Knowledge Check: Why the tags?

Why do we put the report between `<report>` and `</report>`?

- **a.** The model only reads XML
- **b.** It separates our data from our instructions, so the model knows what is data
- **c.** It makes the call cheaper
- **d.** It hides the data from the model

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m4_tags", answer)

#### Knowledge Check: A number that is not in the report

An RM asks the digest bot for BBCA's dividend yield, which is **not** in the report. With our system prompt, what is
the right behaviour?

- **a.** Give the yield from the model's own knowledge
- **b.** Say that the report does not contain it
- **c.** Estimate it from the price
- **d.** Call Sectors to get it

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m4_missing", answer)

# 4.4 Prompt Templates

> **The request**
>
> Your manager: **"Tomorrow there is a new report, and every RM wants a brief on a different bank. Don't copy-paste
> f-strings: make the prompt one thing we can reuse."**

> **In this section**
>
> - **Key question:** How do we write a prompt once and fill it in for any report or bank?
> - **How we run it:** **Live, about 6 minutes.**
> - **What you will do:** turn the 4.3 prompt into a template, see the prompt it makes, and send it to `digest_agent`.
> - **What carries forward:** `DIGEST_TEMPLATE` and `digest_agent`, used in 4.5, the example and Module 9.

A **prompt template** is the prompt with **blanks** in `{curly braces}`. In Python it is just a string: `.format(...)`
fills the blanks and returns the prompt. The rules stay in the instructions of `digest_agent`. No model call yet:

In [ ]:
DIGEST_TEMPLATE = "<report>\n{report}\n</report>\n\n{task}"

filled = DIGEST_TEMPLATE.format(report=report, task="Write the digest: one headline sentence, then one short sentence per bank.")
print(filled)

To call the model, we pass the filled template to `digest_agent.run(...)`. The same template works for any report
and any task, and `asyncio.gather(...)` runs several calls at once. `message_history=` passes the 4.2 conversation
along with any filled template.

In [ ]:
reply = await digest_agent.run(DIGEST_TEMPLATE.format(report=report, task="In one sentence: which bank fell most over 10 days?"))
print(reply.output)

#### Your turn: A brief for any bank

Write `BRIEF_TEMPLATE`, a template with **two** blanks, `{report}` and `{ticker}`: the report between tags and the task
*"In two sentences about {ticker} only: its last close with the date, then its 1-day and 10-day change."* Fill it for
NISP and BMRI and run both with `digest_agent` in one `asyncio.gather` call. The right result: NISP `1,185` on
`2026-10-23`, `+0.42%`, `-4.05%`; BMRI `3,900`, `+1.30%`, `-2.74%`.

In [ ]:
## Your code below



#### Knowledge Check: Why a template?

The digest job runs every morning with a new report. What does the template give us over writing the f-string again?

- **a.** The model answers faster
- **b.** One prompt, written and reviewed once; each run only fills in the blanks
- **c.** The model remembers yesterday's report
- **d.** It removes the need for a system message

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m4_template", answer)

#### Knowledge Check: A curly brace too many

You add an example to a template: `Answer like {"ticker": "BBCA"}`. Now `.format(...)` fails with a `KeyError`.
Why?

- **a.** The model does not accept JSON
- **b.** In a template, `{...}` is a blank to fill; write `{{` and `}}` for a literal brace
- **c.** A template cannot contain examples
- **d.** The report is too long

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m4_braces", answer)

# 4.5 Structured Output

> **The request**
>
> Your manager: **"Module 9 will raise an alert when a bank moves a lot. Code cannot read sentences; give it fields."**

> **In this section**
>
> - **Key question:** How do we get fields we can trust from the model?
> - **How we run it:** **Live, about 8 minutes.**
> - **What you will do:** describe the fields with Pydantic, pass it as `output_type`, then design a schema of your own.
> - **What carries forward:** the `BankDigest` pattern used in the exercise, and the idea of an alert used on Day 2.

A **Pydantic** model describes the fields and their types. `agent.run(prompt, output_type=Model)` puts a Python
object of that class in `result.output`, checked on the way in: a value outside a `Literal` raises an error instead of
reaching your code (Pydantic AI sends the error back to the model and asks again). `Field(description=...)` tells the
model what a field means. The prompt can be a filled template like any other.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class BankDigest(BaseModel):
    ticker: str
    last_close: int
    trend: Literal["up", "down", "flat"] = Field(description="Direction of the 10-day change")
    summary: str = Field(description="One factual sentence from the report")

class MorningDigest(BaseModel):
    headline: str
    banks: list[BankDigest]

prompt = DIGEST_TEMPLATE.format(report=report, task="Write the digest: one headline sentence, then one short sentence per bank.")
result = await digest_agent.run(prompt, output_type=MorningDigest)
for bank in result.output.banks:
    print(bank.ticker, bank.last_close, bank.trend, "|", bank.summary)
result.output.model_dump()["banks"][0]

#### Your turn: Design an alert

Design a schema for alerts: `Alert` with `ticker` (str), `direction` (only `"up"` or `"down"`), `change_pct` (float,
the 1-day change in percent) and `reason` (str, one sentence from the report), and `Alerts` with a list `alerts`. Ask the
model for an alert for every bank whose **1-day** change is larger than 2% in either direction, through `DIGEST_TEMPLATE`
with your own task. The right result is one
alert: `BBCA`, `down`, about `-2.9`.

In [ ]:
## Your code below



#### Knowledge Check: What guarantees the values?

`output_type=MorningDigest` returns `trend="up"`. What guarantees that `trend` can only be up, down or flat?

- **a.** The system prompt
- **b.** The `Literal[...]` type in the Pydantic model, checked when the object is built
- **c.** The model's training
- **d.** Nothing

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m4_literal", answer)

#### Knowledge Check: Who decides the threshold?

In the alert exercise the model decided which banks moved more than 2%. For Module 9's real alerts, what is better?

- **a.** Keep asking the model: it is good at arithmetic
- **b.** Compute the change and the 2% test in code; let the model only write the reason
- **c.** Ask the model twice and compare
- **d.** Let the RMs decide each morning

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m4_threshold", answer)

# Module 4 Wrap-up

| Question | Answer in this module |
|---|---|
| What does the model know? | Only what is in the messages of this call |
| Where are the rules? | The agent's `instructions` (the system message) |
| Where is the data? | Between `<report>` tags in the prompt |
| How do we reuse a prompt? | A template string with `{blanks}`, filled with `.format(...)` |
| How does code use the answer? | `output_type=` + Pydantic |
| Who chose the data? | **Our code** (the Module 3 report) |

**Next:** `day1/examples/m4_example.py` (it reads today's Module 3 report through a template), then the exercise
`day1/exercises/m4_exercise.py`. In Module 5 the **question** starts choosing the data.